# RoboCasa Demo Gallery

一键启动 [RoboCasa](https://robocasa.ai) 厨房场景与任务的 MuJoCo viewer。

_One-click MuJoCo viewer for RoboCasa kitchen scenes & tasks._

**为什么单开一页 / Why a separate notebook**：RoboCasa 的 `setup.py` 把 `mujoco==3.3.1` / `numpy==2.2.5` 硬钉，跟主 `mujoco` env 里其他场景库（dm_control / mjlab / mujoco_playground 都要 mujoco≥3.7/3.8）注定冲突。所以放在独立的 conda env `robocasa` 里关起来养。
_RoboCasa hard-pins `mujoco==3.3.1` / `numpy==2.2.5` in setup.py, colliding with everything in the main `mujoco` env (dm_control / mjlab / mujoco_playground need ≥3.7/3.8). We isolate it in its own conda env._

**机制 / Mechanism**：所有逻辑在 `scripts/robocasa_demo.py` + `scripts/install_robocasa_env.sh`，notebook 只通过 `!python scripts/robocasa_demo.py <cmd>` 调用。一次只能跑一个 demo，新启动前自动 `kill`。

**前置 / Prereqs**:
- `conda` 可用 / `conda` available
- 显示器 (`DISPLAY=:0`)
- ~15 GB 磁盘（厨房资产 ~10 GB + 依赖）/ ~15 GB disk


## 0. 安装 conda env `robocasa` (Setup)

**幂等 / Idempotent**：env / robocasa / 资产 都已存在则跳过。

**做了什么 / What it does**（见 `scripts/install_robocasa_env.sh`）:
1. `conda create -n robocasa python=3.11`
2. `pip install git+https://github.com/ARISE-Initiative/robosuite.git@master`（不是 PyPI 版！RoboCasa 需要最新 master）
3. `pip install -e dependencies/robocasa`
4. 生成 `macros_private.py`，让 `DATASET_BASE_PATH` 指向 `$ROBOCASA_DATA_PATH/datasets`
5. `python -m robocasa.scripts.download_kitchen_assets --type all` （~10 GB，纹理 + 家具 + objaverse 物体）
6. 写 activate hook 持久化 `ROBOCASA_DATA_PATH=~/.cache/robocasa`

首次跑约 15-30 分钟（取决于带宽）。


In [ ]:
# 一键安装 conda env `robocasa` + 所有依赖 + ~23GB 厨房资产。幂等。
# This single command runs scripts/install_robocasa_env.sh which does:
#
#   conda create -c conda-forge -n robocasa python=3.11 -y
#   conda activate robocasa
#   pip install git+https://github.com/ARISE-Initiative/robosuite.git@master
#   pip install -e dependencies/robocasa
#   pip install robosuite_models
#   python -m robocasa.scripts.download_kitchen_assets --type all   # ~23 GB
#   # + write macros_private.py and activate hook (ROBOCASA_DATA_PATH)
#
# 首次跑 15-30 分钟（取决于带宽）。已装好则跳过。
!bash scripts/install_robocasa_env.sh


In [ ]:
!python scripts/robocasa_demo.py status


In [ ]:
!python scripts/robocasa_demo.py list


---
## 1. 厨房场景浏览 (Kitchen Scene Browsing)

加载不同 `(layout, style)` 组合的厨房场景。viewer 启动后机器人停在原地（idle），鼠标拖动视角自由观察。按 `Ctrl-C` 退出。

_Different kitchen layouts × styles. Robot idles; drag mouse to orbit camera. Ctrl-C to quit._


In [ ]:
# 标准厨房 (layout 1, style 1) + PandaOmron 移动操作平台
!python scripts/robocasa_demo.py launch scene:browse


In [ ]:
# 中岛厨房 (layout 3, style 2)
!python scripts/robocasa_demo.py launch scene:island


In [ ]:
# 一字型厨房 (layout 4, style 4)
!python scripts/robocasa_demo.py launch scene:galley


In [ ]:
# 随机布局 + 随机风格（每次 reset 不一样）
!python scripts/robocasa_demo.py launch scene:random


---
## 2. 操作任务 (Manipulation Tasks)

RoboCasa 内置 100+ 个原子任务。下面挑几个代表性的，加载场景 + 任务目标物体。

_RoboCasa ships 100+ atomic tasks. Below: representative picks. Scene + target objects load together._


In [ ]:
# 拾放：从台面到柜子
!python scripts/robocasa_demo.py launch task:pick_place_cab


In [ ]:
# 拾放：从台面到水槽
!python scripts/robocasa_demo.py launch task:pick_place_sink


In [ ]:
# 开柜门
!python scripts/robocasa_demo.py launch task:open_door


In [ ]:
# 关抽屉
!python scripts/robocasa_demo.py launch task:close_drawer


In [ ]:
# 打开炉灶
!python scripts/robocasa_demo.py launch task:turn_on_stove


In [ ]:
# 打开水龙头
!python scripts/robocasa_demo.py launch task:turn_on_faucet


In [ ]:
# 打开微波炉
!python scripts/robocasa_demo.py launch task:microwave


---
## 3. 关于其它机器人 (Other Robots)

理论上 RoboCasa 还支持 **GR1（人形）/ Tiago（移动操作）**，但当前 robosuite master 的 GR1 控制器配置引用了 `WHOLE_BODY_MINK_IK` 控制器类 —— 该类在仓库里**还没实现**，安装 `mink` / `robosuite_models` 都救不了，是上游 bug。Tiago 加载到 `mj_forward` 时报 `FactorizeHessian: rank-deficient sparse Hessian`，也是模型问题。

👉 现阶段 RoboCasa 在本仓的可用机器人是 **PandaOmron**（Franka Panda + Omron LD-60 移动底盘），已经覆盖上面所有场景与任务 demo。等 robosuite 上游把 mink 控制器补全后再开 GR1/Tiago。

_GR1 and Tiago hit upstream robosuite bugs (missing `WHOLE_BODY_MINK_IK` controller class, broken Tiago model). PandaOmron is the only working robot for now._


---
## 4. 收尾 (Cleanup)


In [ ]:
!python scripts/robocasa_demo.py kill


---
## 进一步 / Next Steps

- **训练数据**：`python -m robocasa.scripts.download_datasets --tasks PnPCounterToCab` 下载该任务的 human/robot demos（每个任务几 GB）
- **回放轨迹**：`python -m robocasa.demos.demo_tasks` 选任务后自动下载并回放 demonstrations（交互式）
- **遥操作收集**：`python -m robocasa.demos.demo_teleop --task Kitchen` 键盘控制 PandaOmron，录制自己的 demo
- **训练 policy**：RoboCasa 集成了 Diffusion Policy / π0 / GR00T，见 [官方文档](https://robocasa.ai/docs/introduction/overview.html)
- **场景列表**：60 种 layouts × 12 种 styles = 720 种厨房组合
